In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("cc_data.csv")

#1 The dimensions (number of rows and columns) of the dataset
rows, cols = df.shape

print("Number of rows:", rows)
print("Number of columns:", cols)

#2 Unique values in each categorical variable
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['number']).columns

# Unique value counts for categorical variables
cat_unique_counts = df[categorical_cols].nunique()
print(cat_unique_counts)

#3 The distribution of numerical variables in the dataset
num_description = df[numerical_cols].describe()
print(num_description)

#4 Checking any missing values in the dataset
missing_counts = df.isnull().sum()
missing_any = missing_counts[missing_counts > 0]

print(missing_any) #no missing values

#5 The summary statistics (mean, median, min, max, etc.) for numerical variables
numerical_cols = df.select_dtypes(include=['number']).columns
num_summary = df[numerical_cols].describe().T

print(num_summary.head())


#6 Correlation between numerical variables?
numerical_cols = df.select_dtypes(include=['number']).columns
corr_matrix = df[numerical_cols].corr()

print(corr_matrix)

# Most correlations are very close to 0,so little to no linear relationship between most pairs of numerical variables.
# lat vs merch_lat and long vs merch_long -very strong correlation
# zip vs long                             -very strong negative correlation
# amt vs is_fraud                         -moderate positive correlation but not sufficient on its own

#7 The distribution of an amt differ across is_fraud categories
import seaborn as sns
import matplotlib.pyplot as plt

# Basic descriptive stats of amt by is_fraud
amt_by_fraud = df.groupby('is_fraud')['amt'].describe()
print(amt_by_fraud)

# Plot distributions (log scale to handle heavy tail)
plt.figure(figsize=(8,5))
sns.kdeplot(data=df, x='amt', hue='is_fraud', common_norm=False, log_scale=True)
plt.title('Distribution of Transaction Amount by Fraud Status (log-scale x)')
plt.xlabel('Transaction Amount (log scale)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

#8 Check for outliers in city_pop and amt using simple IQR method and show basic distributions

cols_to_check = ['city_pop', 'amt']
outlier_info = {}

for col in cols_to_check:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    outlier_info[col] = {
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_bound': lower,
        'upper_bound': upper,
        'num_outliers': outliers.shape[0]
    }

print(outlier_info)

# Visualize with boxplots (on log scale for amt)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.boxplot(y=df['city_pop'])
plt.title('City Population Boxplot')

plt.subplot(1,2,2)
sns.boxplot(y=df['amt'])
plt.yscale('log')
plt.title('Transaction Amount Boxplot (log scale)')

plt.tight_layout()
plt.show()

#city_pop - there is a large number of outlier cities on the high end of population
#amt      - is also heavily right‑skewed, with many high-value transactions flagged as outliers

#9 Trends or patterns in the data over time

# Inspect columns to guess time fields
print(df.columns)

# If there is a 'trans_date_trans_time' column, convert it to datetime and extract parts
if 'trans_date_trans_time' in df.columns:
    df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'])
    df['trans_date'] = df['trans_dt'].dt.date
    df['trans_hour'] = df['trans_dt'].dt.hour
    df['trans_dow'] = df['trans_dt'].dt.dayofweek
else:
    # If no clear datetime column, just skip further plots
    print('No explicit datetime column found to analyze trends over time.')

# Only proceed if we created trans_date
if 'trans_date' in df.columns:
    # Daily counts overall and fraud
    daily = df.groupby('trans_date').agg(
        n_txn=('is_fraud', 'size'),
        n_fraud=('is_fraud', 'sum')
    ).reset_index()
    daily['fraud_rate'] = daily['n_fraud'] / daily['n_txn']
    print(daily.head())

#Daily trends (over calendar time)
plt.figure(figsize=(12,4))
sns.lineplot(data=daily, x='trans_date', y='n_txn')
plt.title('Number of Transactions per Day')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))
sns.lineplot(data=daily, x='trans_date', y='fraud_rate')
plt.title('Daily Fraud Rate Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#Transaction counts clearly vary over time (some days busier than others)
#Fraud rate is low overall but fluctuates day-to-day; some dates have noticeable bumps in fraud rate compared to others
#There is no single explosive spike dominating everything, but the rate isn’t perfectly flat

# Hour-of-day pattern
hourly = df.groupby('trans_hour').agg(
    n_txn=('is_fraud', 'size'),
    n_fraud=('is_fraud', 'sum')
    ).reset_index()
hourly['fraud_rate'] = hourly['n_fraud'] / hourly['n_txn']
print(hourly)

plt.figure(figsize=(10,4))
sns.lineplot(data=hourly, x='trans_hour', y='n_txn')
plt.title('Transactions by Hour of Day')
plt.xticks(range(0,24))
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
sns.lineplot(data=hourly, x='trans_hour', y='fraud_rate')
plt.title('Fraud Rate by Hour of Day')
plt.xticks(range(0,24))
plt.tight_layout()
plt.show()

#Transaction volume follows a typical daily rhythm: certain hours (often daytime/evening) are busier than others
#Fraud rate by hour shows some variation; some hours have consistently higher fraud rates than others (even if total volume is lower)
#Fraud is not evenly distributed across the 24 hours

# Day-of-week pattern
dow = df.groupby('trans_dow').agg(
    n_txn=('is_fraud', 'size'),
    n_fraud=('is_fraud', 'sum')
).reset_index()
dow['fraud_rate'] = dow['n_fraud'] / dow['n_txn']
print(dow)

plt.figure(figsize=(8,4))
sns.barplot(data=dow, x='trans_dow', y='n_txn')
plt.title('Transactions by Day of Week (0=Mon)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,4))
sns.barplot(data=dow, x='trans_dow', y='fraud_rate')
plt.title('Fraud Rate by Day of Week (0=Mon)')
plt.tight_layout()
plt.show()

#Some weekdays have noticeably more transactions than others (e.g., typical midweek vs weekend differences)
#Fraud rate also varies by weekday; certain days have higher fraud proportion even if volume is similar

#10 Explore how the target variable `is_fraud` distributes across key categorical features

# Quick check of target balance overall
fraud_counts = df['is_fraud'].value_counts(normalize=False).rename('count')
fraud_rates = df['is_fraud'].value_counts(normalize=True).rename('rate')
fraud_summary = pd.concat([fraud_counts, fraud_rates], axis=1)
print(fraud_summary)

# Helper to compute fraud rate by category for a given column
cat_cols = ['category', 'gender', 'state']
summary_dict = {}
for col in cat_cols:
    grp = df.groupby(col)['is_fraud'].agg(['count', 'sum'])
    grp['fraud_rate'] = grp['sum'] / grp['count']
    grp = grp.sort_values('fraud_rate', ascending=False)
    summary_dict[col] = grp.head(10)
    print('\
Top 10 categories for ' + col + ' by fraud rate:')
    print(summary_dict[col].head())

# Plot fraud rate by transaction category (top 10 by volume)
cat_by_vol = df['category'].value_counts().head(10).index
cat_rate = df[df['category'].isin(cat_by_vol)].groupby('category')['is_fraud'].mean().reset_index()

plt.figure(figsize=(10,4))
sns.barplot(data=cat_rate, x='category', y='is_fraud')
plt.xticks(rotation=45, ha='right')
plt.title('Fraud Rate by Transaction Category (Top 10 by Volume)')
plt.tight_layout()
plt.show()

# Online-related categories like shopping_net, misc_net, and grocery_pos have noticeably higher fraud rates
# Point-of-sale or in-person categories (e.g. shopping_pos, gas_transport, etc.) generally have lower fraud proportions
# This indicates transaction type/category is very informative for predicting fraud

# Plot fraud rate by gender
gender_rate = df.groupby('gender')['is_fraud'].mean().reset_index()
plt.figure(figsize=(4,4))
sns.barplot(data=gender_rate, x='gender', y='is_fraud')
plt.title('Fraud Rate by Gender')
plt.tight_layout()
plt.show()

# Both genders have similar fraud rates, with a slight difference (males a bit higher in this sample)
# Gender may have some signal, but much weaker than transaction category

# Plot fraud rate by state (top 10 by volume)
state_by_vol = df['state'].value_counts().head(10).index
state_rate = df[df['state'].isin(state_by_vol)].groupby('state')['is_fraud'].mean().reset_index()
plt.figure(figsize=(10,4))
sns.barplot(data=state_rate, x='state', y='is_fraud')
plt.title('Fraud Rate by State (Top 10 by Volume)')
plt.tight_layout()
plt.show()

# Some states with very few transactions can show an artificially high fraud rate
# More populous states (with many transactions) have lower but more stable estimates of fraud rate

#11 Quick data quality scan for unusual or unexpected values in the dataset
import numpy as np

#Basic info on nulls
null_counts = df.isnull().sum()
print(null_counts)

#Check for duplicates by transaction id
if 'trans_num' in df.columns:
    dup_trans = df['trans_num'].duplicated().sum()
    print('Duplicate trans_num count:')
    print(dup_trans)

#Basic numeric sanity checks
num_cols = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long']
num_summary = df[num_cols].describe(percentiles=[0.01, 0.5, 0.99])
print(num_summary)

#Look for impossible or suspicious values
suspicious = {'non_positive_amt': df[df['amt'] <= 0].shape[0],
              'invalid_lat': df[(df['lat'] < -90) | (df['lat'] > 90)].shape[0],
              'invalid_long': df[(df['long'] < -180) | (df['long'] > 180)].shape[0],
              'invalid_merch_lat': df[(df['merch_lat'] < -90) | (df['merch_lat'] > 90)].shape[0],
              'invalid_merch_long': df[(df['merch_long'] < -180) | (df['merch_long'] > 180)].shape[0],
              'non_positive_city_pop': df[df['city_pop'] <= 0].shape[0]}
print(suspicious)

#Check a few categorical oddities
print(df['gender'].value_counts(dropna=False))
print(df['category'].value_counts().head())
print(df['state'].value_counts().head())

#There are no obvious data quality errors(no nulls, invalid coordinates, negative amounts, or duplicate transaction IDs)

#12 Any potential data entry errors or inconsistencies in the dataset
#There are no clear-cut data entry errors like:
#                   Wrong column types,
#                   Impossible coordinates,
#                   Negative or zero amounts,
#                   Missing/duplicated transaction IDs.

#13  The distribution of numerical variables vary between different groups or segments of the dataset

#Compare fraud vs non-fraud for transaction amount
plt.figure(figsize=(6,4))
sns.kdeplot(data=df, x='amt', hue='is_fraud', common_norm=False, log_scale=True)
plt.title('Amount distribution by fraud flag (log scale)')
plt.tight_layout()
plt.show()

#Fraud transactions tend to have higher amounts on average and a heavier upper tail.
#Non-fraud is more concentrated at lower amounts, with fewer extreme highs.
#Fraud’s density shifted to the right compared to non-fraud.

#Summary stats by fraud flag for a few numerical columns
num_cols = ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long']
fraud_group_stats = df.groupby('is_fraud')[num_cols].describe().T
print(fraud_group_stats.head(24))

#Compare amount distribution by gender (non-fraud only)
plt.figure(figsize=(6,4))
sns.kdeplot(data=df[df['is_fraud']==0], x='amt', hue='gender', common_norm=False, log_scale=True)
plt.title('Amount distribution by gender (non-fraud only, log scale)')
plt.tight_layout()
plt.show()

#The shape for M vs F is very similar.
#Any differences are minor relative to the fraud vs non-fraud contrast; gender does not massively shift the amount distribution.

#Compare amount distribution by a few top categories
top_cats = df['category'].value_counts().head(4).index
subset = df[df['category'].isin(top_cats)]
plt.figure(figsize=(7,4))
sns.boxplot(data=subset, x='category', y='amt')
plt.yscale('log')
plt.title('Amount distribution by top categories (log y)')
plt.tight_layout()
plt.show()

#Different categories have distinctly different amount scales:
#         Some categories (e.g. large-ticket types like certain services/shopping) have higher medians and wider spread.
#         Others (e.g. everyday spending like small retail) are tightly clustered at low amounts.
#So category is an important segment: amt is not identically distributed across them.

#City population by fraud flag
plt.figure(figsize=(6,4))
sns.kdeplot(data=df, x='city_pop', hue='is_fraud', common_norm=False, log_scale=True)
plt.title('City population distribution by fraud flag (log scale)')
plt.tight_layout()
plt.show()

#Both fraud and non-fraud occur across the full range of city sizes — from very small towns up to big cities.
#The shapes are relatively similar, suggesting city size has less impact on whether a transaction is fraudulent than amount does.
#There are lots of very small-population cities in both groups, which aligns with the earlier note about many rural locations.

#14 Compute simple feature importance and associations with the target
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

target = 'is_fraud'
X = df.drop(columns=[target])
y = df[target]

# Identify basic numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Simple preprocessing + logistic regression for interpretability
preprocess = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)
log_reg = LogisticRegression(max_iter=1000, n_jobs=-1)
clf = Pipeline(steps=[('preprocess', preprocess), ('model', log_reg)])
clf.fit(X, y)

# Get AUC to ensure model is at least reasonable
probs = clf.predict_proba(X)[:,1]
auc = roc_auc_score(y, probs)
print(auc)

#The top factors that influence the target variable
#   -Transaction amount (amt)
#   -Transaction category, especially card-not-present / online vs in-person
#   -Temporal features (hour, day-of-week)
#   -Geographic relationships between customer and merchant (via coordinates)
#   -Demographic/contextual variables (gender, state, job, age) – secondary




























